# Example power flow problem
The example is taken from the tutorials in the rhtlab repository ("00_welcome_to_python").


In [1]:
import numpy as np
import pandas as pd
import pandapower as pp


# Creating an empty pandapower network
net_ExPF1_Pb1 = pp.create_empty_network(
    name="ExPF1_Pb1", f_hz=50.0, sn_mva=100, add_stdtypes=True
)

# create buses
b1 = pp.create_bus(net_ExPF1_Pb1, vn_kv=400, name="Bus 1")
b2 = pp.create_bus(net_ExPF1_Pb1, vn_kv=400, name="Bus 2")
b3 = pp.create_bus(net_ExPF1_Pb1, vn_kv=400, name="Bus 3")
b4 = pp.create_bus(net_ExPF1_Pb1, vn_kv=400, name="Bus 4")
b5 = pp.create_bus(net_ExPF1_Pb1, vn_kv=20, name="Bus 5")

# create load
load_2 = pp.create_load(net_ExPF1_Pb1, bus=b2, p_mw=100, q_mvar=20, name="Load 2")
load_3 = pp.create_load(net_ExPF1_Pb1, bus=b3, p_mw=200, q_mvar=80, name="Load 3")

# create slack bus and generators (the external grid can be seen as a generator connected to slack bus)
gen_1 = pp.create_ext_grid(
    net_ExPF1_Pb1, bus=b1, vm_pu=1.0, min_q_mvar=-100, max_q_mvar=100, name="Gen 1"
)
# pp.create_gen(net,bus=b1,p_mw=0,vm_pu=1,min_q_mvar=-100,max_q_mvar=100,name="Gen 1",slack=True)

# If no name is given, pandapower still assigns an index ID to each element (in this case the two generators)
pp.create_gen(
    net_ExPF1_Pb1,
    bus=b2,
    p_mw=150,
    vm_pu=1.05,
    min_q_mvar=-200,
    max_q_mvar=200,
    name="Gen 2",
)
pp.create_gen(
    net_ExPF1_Pb1,
    bus=b5,
    p_mw=100,
    vm_pu=1,
    min_q_mvar=-100,
    max_q_mvar=100,
    name="Gen 5",
)

# create transformer
trafo = pp.create_transformer_from_parameters(
    net_ExPF1_Pb1,
    hv_bus=b4,
    lv_bus=b5,
    sn_mva=200,
    vn_hv_kv=400,
    vn_lv_kv=20,
    vk_percent=10,
    vkr_percent=0,
    pfe_kw=0,
    i0_percent=0,
    name="Transfo T1",
)

# Evaluation of base values and definition of the dataframe containing line parameters
S_base = 100e6
U_base = 400e3
Z_base = U_base**2 / S_base
Y_base = 1 / Z_base
Ligne_value = {
    "R_pu": [0.02, 0.02, 0.02, 0.04, 0.04],
    "X_pu": [0.2, 0.4, 0.4, 0.4, 0.4],
    "B_pu": [0.8, 0.8, 0.4, 0.4, 0.4],
    "Un": [400, 400, 400, 400, 400],
}
DF_Ligne = pd.DataFrame(Ligne_value, index=["A", "B", "C", "D", "E"])
Z_base

1600.0

In [2]:
# The parameters of the lines are calculatd in electrical units (not in per unit)
rl_ohm_per_km = DF_Ligne.R_pu * Z_base
xl_ohm_per_km = DF_Ligne.X_pu * Z_base
cl_nF_per_km = 1 / (DF_Ligne.B_pu * Z_base**2 * 2 * np.pi * 50) * 10**9
print("Line resistance :\n", rl_ohm_per_km)
print("Line impedance :\n", xl_ohm_per_km)
print("Line capacitance :\n", cl_nF_per_km)
line_parameter = {
    "r_ohm_per_km": rl_ohm_per_km,
    "x_ohm_per_km": xl_ohm_per_km,
    "c_nf_per_km": cl_nF_per_km,
}

Line resistance :
 A    32.0
B    32.0
C    32.0
D    64.0
E    64.0
Name: R_pu, dtype: float64
Line impedance :
 A    320.0
B    640.0
C    640.0
D    640.0
E    640.0
Name: X_pu, dtype: float64
Line capacitance :
 A    1.554247
B    1.554247
C    3.108495
D    3.108495
E    3.108495
Name: B_pu, dtype: float64


In [ ]:
# create line
line_a = pp.create_line_from_parameters(
    net_ExPF1_Pb1,
    from_bus=b1,
    to_bus=b2,
    length_km=1,
    name="Line A",
    r_ohm_per_km=rl_ohm_per_km.A,
    x_ohm_per_km=xl_ohm_per_km.A,
    c_nf_per_km=cl_nF_per_km.A,
    max_i_ka=1,
)
line_b = pp.create_line_from_parameters(
    net_ExPF1_Pb1,
    from_bus=b2,
    to_bus=b3,
    length_km=1,
    name="Line B",
    r_ohm_per_km=rl_ohm_per_km.B,
    x_ohm_per_km=xl_ohm_per_km.B,
    c_nf_per_km=cl_nF_per_km.B,
    max_i_ka=1,
)
line_c = pp.create_line_from_parameters(
    net_ExPF1_Pb1,
    from_bus=b1,
    to_bus=b3,
    length_km=1,
    name="Line C",
    r_ohm_per_km=rl_ohm_per_km.C,
    x_ohm_per_km=xl_ohm_per_km.C,
    c_nf_per_km=cl_nF_per_km.C,
    max_i_ka=1,
)
line_d = pp.create_line_from_parameters(
    net_ExPF1_Pb1,
    from_bus=b3,
    to_bus=b4,
    length_km=1,
    name="Line D",
    r_ohm_per_km=rl_ohm_per_km.D,
    x_ohm_per_km=xl_ohm_per_km.D,
    c_nf_per_km=cl_nF_per_km.D,
    max_i_ka=1,
)
line_e = pp.create_line_from_parameters(
    net_ExPF1_Pb1,
    from_bus=b2,
    to_bus=b4,
    length_km=1,
    name="Line E",
    r_ohm_per_km=rl_ohm_per_km.E,
    x_ohm_per_km=xl_ohm_per_km.E,
    c_nf_per_km=cl_nF_per_km.E,
    max_i_ka=1,
)

# Checking the lines
net_ExPF1_Pb1.line

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service
0,Line A,None,0,1,1.0,32.0,320.0,1.554247,0.0,1.0,1.0,1,None,True
1,Line B,None,1,2,1.0,32.0,640.0,1.554247,0.0,1.0,1.0,1,None,True
2,Line C,None,0,2,1.0,32.0,640.0,3.108495,0.0,1.0,1.0,1,None,True
3,Line D,None,2,3,1.0,64.0,640.0,3.108495,0.0,1.0,1.0,1,None,True
4,Line E,None,1,3,1.0,64.0,640.0,3.108495,0.0,1.0,1.0,1,None,True


In [4]:
# Information about the network
net_ExPF1_Pb1

This pandapower network includes the following parameter tables:
   - bus (5 elements)
   - load (2 elements)
   - gen (2 elements)
   - ext_grid (1 element)
   - line (5 elements)
   - trafo (1 element)

In [5]:
# Information about the tranformer
net_ExPF1_Pb1.trafo

,name,std_type,hv_bus,lv_bus,sn_mva,vn_hv_kv,vn_lv_kv,vk_percent,vkr_percent,pfe_kw,i0_percent,shift_degree,tap_side,tap_neutral,tap_min,tap_max,tap_step_percent,tap_step_degree,tap_pos,tap_phase_shifter,parallel,df,in_service
0,Transfo T1,None,3,4,200.0,400.0,20.0,10.0,0.0,0.0,0.0,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,False,1,1.0,True


In [6]:
# Solution of the network
pp.runpp(net_ExPF1_Pb1)
net_ExPF1_Pb1

This pandapower network includes the following parameter tables:
   - bus (5 elements)
   - load (2 elements)
   - gen (2 elements)
   - ext_grid (1 element)
   - line (5 elements)
   - trafo (1 element)
 and the following results tables:
   - res_bus (5 elements)
   - res_line (5 elements)
   - res_trafo (1 element)
   - res_ext_grid (1 element)
   - res_load (2 elements)
   - res_gen (2 elements)

In [7]:
net_ExPF1_Pb1.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.000000,0.000000,-57.252001,-31.160412
1,1.050000,0.194014,-50.000000,-118.481567
2,0.799545,-17.075981,200.000000,80.000000
3,0.981705,5.170810,0.000000,0.000000
4,1.000000,8.090251,-100.000000,-39.139051


In [8]:
net_ExPF1_Pb1.res_line.pl_mw.sum()

7.252000890487759

In [9]:
net_ExPF1_Pb1.res_gen

,p_mw,q_mvar,va_degree,vm_pu
0,150.0,138.481567,0.194014,1.05
1,100.0,39.139051,8.090251,1.00


In [10]:
net_ExPF1_Pb1.res_ext_grid

,p_mw,q_mvar
0,57.252001,31.160412


In [11]:
# Admittance matrix
pd.DataFrame(net_ExPF1_Pb1._ppc["internal"]["Ybus"].toarray()).round(1)

,0,1,2,3,4
0,0.6-7.4j,-0.5+5.0j,-0.1+2.5j,0.0+ 0.0j,0.0+ 0.0j
1,-0.5+5.0j,0.9-9.9j,-0.1+2.5j,-0.2+ 2.5j,0.0+ 0.0j
2,-0.1+2.5j,-0.1+2.5j,0.5-7.5j,-0.2+ 2.5j,0.0+ 0.0j
3,0.0+0.0j,-0.2+2.5j,-0.2+2.5j,0.5-24.9j,0.0+20.0j
4,0.0+0.0j,0.0+0.0j,0.0+0.0j,0.0+20.0j,0.0-20.0j
